## Setup
Run notebooks 01–07 first. Install dependencies once from the project root with `python -m pip install -r requirements.txt`.

In [11]:
from pathlib import Path
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate/"data").is_dir() and (candidate/"notebooks").is_dir(): PROJECT_ROOT=candidate; break
else: raise RuntimeError("Start Jupyter inside the project so its root can be found.")
DATA_DIR=PROJECT_ROOT/"data"; RAW_DATA_DIR=DATA_DIR/"01_raw"; INTERMEDIATE_DIR=DATA_DIR/"02_intermediate"
MODEL_DIR=PROJECT_ROOT/"models"; REPORT_DIR=PROJECT_ROOT/"reports"
INTERMEDIATE_DIR.mkdir(parents=True,exist_ok=True); MODEL_DIR.mkdir(parents=True,exist_ok=True); REPORT_DIR.mkdir(parents=True,exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")

Project root: c:\Users\acer\Desktop\Programs\Projects_for _data_science\Fraud_Detection_Model


# 09 · Model explainability

This notebook explains global patterns and selected individual predictions with SHAP. Attributions describe how inputs move the model output relative to a chosen background. They do not establish causation, prove fraud, or explain information the model never received.

### SHAP installation (one time)

The project pins `shap==0.48.0` in `requirements.txt`. Install it into the **same Python environment as this notebook kernel** using `%pip install shap==0.48.0`, then restart the kernel. To install all project dependencies from the repository root, use `python -m pip install -r requirements.txt`. Do not run an install command every time you execute the notebook.


## 1. Load the model and validation rows

### Why reload the exact model inputs

Explanations are meaningful only when the row has the same columns, ordering, and preprocessing as model training. The chronological validation rows make the examples comparable to notebook 07. Applying the artifact’s imputer/scaler, if present, reproduces the model’s input space.

In [ ]:
train_features = pd.read_pickle(INTERMEDIATE_DIR / "train_features.pkl")
artifact = joblib.load(MODEL_DIR / "best_experiment_model.joblib")
feature_cols = list(artifact["feature_cols"])

ordered = train_features.sort_values("TransactionDT", kind="stable").reset_index(drop=True)
validation = ordered.iloc[int(len(ordered) * 0.8):].copy()
X_validation = validation[feature_cols].replace([np.inf, -np.inf], np.nan)

# Apply the same fitted transformations that the model uses at prediction time.
if artifact.get("imputer") is not None:
    X_validation = pd.DataFrame(
        artifact["imputer"].transform(X_validation),
        columns=feature_cols,
        index=validation.index,
    )
if artifact.get("scaler") is not None:
    X_validation = pd.DataFrame(
        artifact["scaler"].transform(X_validation),
        columns=feature_cols,
        index=validation.index,
    )

model = artifact["model"]
X = X_validation
print(f"Model: {type(model).__name__} | validation matrix: {X.shape}")

## 2. Build explanations on a fixed, bounded sample

The fixed seed makes the sample repeatable. A modest background and sample keep runtime manageable. The background population defines what “relative to typical” means in these plots.

For XGBoost, feature contributions come from XGBoost’s native TreeSHAP prediction output, which avoids SHAP’s compiled extension. Those values are in raw-margin (log-odds) space. Other supported classifiers use SHAP’s model-agnostic permutation explainer; it is slower, so the notebook limits its sample size.


### Why this explainer depends on the model family

For the XGBoost winner, `pred_contribs=True` returns one feature contribution per input plus a bias term. These add to the raw model margin (log-odds), not directly to probability. This route avoids SHAP’s blocked compiled extension. The fallback permutation explainer calls the model as a function and is much slower, so it uses only 25 rows and a small reference background.

In [ ]:
# Use a seeded sample so reruns explain the same rows.
rng = np.random.default_rng(42)

if type(model).__name__ == "XGBClassifier":
    SAMPLE_SIZE = min(500, len(X))
    sample_positions = rng.choice(len(X), size=SAMPLE_SIZE, replace=False)
    X_sample = X.iloc[sample_positions]

    import xgboost as xgb

    dmatrix = xgb.DMatrix(X_sample, feature_names=list(X_sample.columns))
    contributions = model.get_booster().predict(dmatrix, pred_contribs=True)
    # XGBoost returns one column per feature plus a final bias/base-value column.
    explanation = shap.Explanation(
        values=contributions[:, :-1],
        base_values=contributions[:, -1],
        data=X_sample.to_numpy(),
        feature_names=list(X_sample.columns),
    )
    explainer_name = "XGBoost native TreeSHAP"
else:
    # Model-agnostic explanations are more expensive, so use a small sample.
    SAMPLE_SIZE = min(25, len(X))
    sample_positions = rng.choice(len(X), size=SAMPLE_SIZE, replace=False)
    X_sample = X.iloc[sample_positions]
    reference = X.iloc[[0]]
    explainer = shap.Explainer(
        model.predict_proba,
        reference,
        algorithm="permutation",
    )
    explanation = explainer(
        X_sample,
        max_evals=2 * X_sample.shape[1] + 1,
    )
    explainer_name = "SHAP permutation explainer"

print(f"Explainer: {explainer_name}")
print(f"Explanation array shape: {explanation.values.shape}")

## 3. Global feature influence

### How to read the global plots

The beeswarm shows one point per explained row and feature: horizontal position is contribution direction and size; color is the feature value. The bar plot ranks mean absolute contribution, so it answers “which inputs move predictions most on this sample?” but hides whether each feature raises or lowers risk. Correlated features can share attribution.

In [ ]:
values = explanation

# Some classifier explainers return an output axis for both classes.
# Select class 1, the fraud class, before drawing global or local plots.
if values.values.ndim == 3:
    positive_class = 1
    base_values = np.asarray(values.base_values)
    if base_values.ndim == 2:
        base_values = base_values[:, positive_class]
    elif base_values.ndim == 1 and len(base_values) > positive_class:
        base_values = base_values[positive_class]

    values = shap.Explanation(
        values=values.values[:, :, positive_class],
        base_values=base_values,
        data=values.data,
        feature_names=values.feature_names,
    )

shap.plots.beeswarm(values, max_display=20)

### Why show a second global view

The bar plot reduces the beeswarm to average absolute contribution. It is easier to rank, but loses the distribution and direction of effects. Use it for a compact overview; use the beeswarm to see whether a feature has mixed behavior across transactions.

In [ ]:
shap.plots.bar(values, max_display=20)

## 4. Explain one high-scoring example

For XGBoost the waterfall adds up to a raw margin (log-odds); the printed model probability is shown separately.


### Why inspect a local explanation

The highest-scored sampled row is an intentionally salient case for learning how contributions combine. The waterfall starts at a reference output and adds each feature’s contribution for this row. It is not a representative case or proof of wrongdoing; inspect ordinary and mistaken cases too.

In [ ]:
sample_probabilities = model.predict_proba(X_sample)[:, 1]
local_position = int(np.argmax(sample_probabilities))
print(f"Selected row's fraud probability: {sample_probabilities[local_position]:.4f}")
shap.plots.waterfall(values[local_position], max_display=15)

## 5. Review limits

Correlated inputs can share attributions. A strong feature may be a proxy or data-quality artifact, so verify its source and availability at decision time. Explanations support analyst review; they are not evidence by themselves. Record model version, explainer version, and reference population when operationalizing explanations.